# Xenium spatial proximity analysis: DCM and IHD

**Purpose.** Analyze the spatial proximity of VCAN-, DCN- and PDGFRA-expressing fibroblasts to NPPA-high and NPPA-negative cardiomyocytes, and reproduce the broad ECM/fibroblast-marker delta-distance plots supplied for DCM and IHD.

## Dataset and annotation versions

- **DCM:** `DCM_final.h5ad`, annotation column `celltype_corrected`.
- **IHD:** `adata_celltype_final_corrected_v4.h5ad`, annotation column `celltype_final_corrected_v4`.

The IHD annotation object and cell-type column specified above correspond to the cell populations used for the supplied spatial summaries.

## Method

Cells were grouped initially by BANKSY spatial clustering and assigned to broad cell classes using established marker genes. Compartment-specific Scanpy/Leiden reclustering and manual marker-based reannotation produced the final cell-type labels used here. All downstream expression analyses use total-count-normalized (`target_sum=10,000`) and `log1p`-transformed expression. Raw counts are taken from `layers['counts']` when present; otherwise an existing `log_norm` layer is accepted, or `.X` is treated as raw counts only after validation. Distances use the first two spatial coordinates as stored in the AnnData object; no coordinate rescaling is applied, so distance units are the coordinate units in the input file.

For each target marker, marker-high and marker-low fibroblasts are the top and bottom 10% among marker-detectable fibroblasts; marker-negative fibroblasts have zero expression. The NPPA-high anchor is the top 10% of NPPA-positive cardiomyocytes. The NPPA-negative anchor contains cardiomyocytes with zero NPPA expression. Euclidean distance to the nearest anchor cell is calculated with `scipy.spatial.cKDTree`.

Targeted marker-high versus marker-low/negative comparisons use one-sided Mann–Whitney U tests (`alternative="less"`) with Benjamini–Hochberg FDR correction within each dataset and anchor. For the random-control comparison, 100 size-matched fibroblast samples are drawn; the reported p-value is the median of the 100 one-sided Mann–Whitney U p-values. The random-control distance summary is the mean of the 100 sampled median distances, with the 5th and 95th percentiles also saved. The broad ECM-marker screen follows the directional hypothesis in the supplied figures and tests whether marker-high fibroblasts are closer than marker-low fibroblasts (one-sided Mann–Whitney U; FDR correction across tested genes).

**Delta conventions differ by analysis:** targeted comparisons use `distance(marker-high) − distance(comparator)`, so negative values mean marker-high FB are closer. The broad ECM summary figures use `distance(marker-low) − distance(marker-high)`, so positive values mean marker-high FB are closer.

**Inference note.** These tests treat cells as observations. If multiple donors, tissue sections or regions contribute cells, cell-level tests do not account for within-sample dependence; sample-aware inference should be used where the study design and sample identifiers permit it.

**Random controls.** Random-control sampling uses seed 42 for both DCM and IHD, matching the supplied spatial-analysis workflows.

## Cell counts and NPPA anchor definitions

| Dataset | CM | FB | NPPA-positive CM | NPPA-negative CM | NPPA-high anchor (top 10% of NPPA-positive CM) |
|---|---:|---:|---:|---:|---:|
| DCM | 66,162 | 46,406 | 39,347 | 26,815 | 3,935 |
| IHD | 43,961 | 27,223 | 29,984 | 13,977 | 2,999 |

These values are the expected counts for the annotation versions specified above. The code recomputes them from the loaded objects and stops if any value differs; this avoids silently running against a different annotation version or expression layer.

## Reported targeted-distance results: NPPA-high CM anchor

The following medians and deltas are transcribed from the reported result summary (not recalculated by this notebook at authoring time). In these tables, **Δ = median distance(marker-high FB) − median distance(comparator FB)**; negative Δ means closer proximity for marker-high fibroblasts.

### DCM — NPPA-high cardiomyocyte anchor

| Marker | High median | Low median (Δ high − low) | Marker-negative median (Δ high − negative) | Random median (Δ high − random) |
|---|---:|---:|---:|---:|
| VCAN | 82.6 | 145.3 (−62.7) | 130.8 (−48.3) | 119.3 (−36.7) |
| DCN | 95.9 | 143.7 (−47.8) | 94.1 (+1.8) | 119.2 (−23.3) |
| PDGFRA | 122.9 | 109.9 (+13.0) | 109.6 (+13.2) | 119.1 (+3.8) |

### IHD — NPPA-high cardiomyocyte anchor

| Marker | High median | Low median (Δ high − low) | Marker-negative median (Δ high − negative) | Random median (Δ high − random) |
|---|---:|---:|---:|---:|
| VCAN | 82.5 | 311.8 (−229.3) | 319.6 (−237.1) | 145.4 (−62.9) |
| DCN | 106.3 | 187.4 (−81.1) | 361.5 (−255.2) | 146.8 (−40.5) |
| PDGFRA | 301.3 | 74.0 (+227.3) | 159.8 (+141.5) | 144.8 (+156.5) |

## Reported targeted-distance results: NPPA-negative CM anchor (DCM and IHD)

Here the anchor group is NPPA-negative cardiomyocytes. Delta is again `distance(marker-high FB) − distance(comparator FB)`, so negative Δ means marker-high fibroblasts are closer to NPPA-negative CM.

### DCM — NPPA-negative cardiomyocyte anchor

| Marker | High median | Low median (Δ high − low) | Marker-negative median (Δ high − negative) | Random median (Δ high − random) |
|---|---:|---:|---:|---:|
| VCAN | 58.4 | 44.7 (+13.6) | 45.2 (+13.2) | 47.2 (+11.2) |
| DCN | 53.1 | 42.6 (+10.5) | 80.5 (−27.4) | 47.1 (+5.9) |
| PDGFRA | 45.1 | 51.3 (−6.2) | 49.1 (−4.0) | 47.2 (−2.1) |

### IHD — NPPA-negative cardiomyocyte anchor

| Marker | High median | Low median (Δ high − low) | Marker-negative median (Δ high − negative) | Random median (Δ high − random) |
|---|---:|---:|---:|---:|
| VCAN | 250.4 | 82.7 (+167.6) | 117.8 (+132.6) | 154.4 (+96.0) |
| DCN | 172.2 | 169.7 (+2.5) | 135.6 (+36.6) | 154.6 (+17.6) |
| PDGFRA | 121.4 | 211.9 (−90.5) | 159.6 (−38.2) | 154.9 (−33.4) |

*Rounding note: the Δ values are transcribed as reported. Differences of 0.1 from subtraction of the displayed rounded medians can arise when deltas were calculated from unrounded medians.*

## Interpretation of the reported targeted results

Across DCM and IHD, VCAN-high fibroblasts show the clearest and most consistent pattern: substantially shorter median distances to the NPPA-high CM anchor than marker-low, marker-negative and random fibroblasts, and greater distances from NPPA-negative CM. DCN has a less specific/context-dependent pattern, whereas PDGFRA does not show enrichment near NPPA-high CM and is markedly more distant in IHD. These summaries are the supplied descriptive results; the the notebook calculates the p-values and FDR-adjusted values when the `.h5ad` inputs are available.

No p-values were supplied alongside the median values above, so the notebook does not infer or insert statistical significance for those exact reported medians.

## Supplied broad ECM / fibroblast-marker summary figures

DCM summary plot: positive delta means marker-high FB are closer to NPPA-high CM.

In [ ]:
from IPython.display import IFrame, display
from pathlib import Path

figure_path = Path("reference_figures/ECM_FB_marker_delta_distances_DCM_summary_plot.pdf")
if figure_path.exists():
    display(IFrame(src=str(figure_path), width="100%", height=600))
else:
    print(f"Original figure not found: {figure_path.resolve()}")

IHD summary plot: positive delta means marker-high FB are closer to NPPA-high CM.

In [ ]:
from IPython.display import IFrame, display
from pathlib import Path

figure_path = Path("reference_figures/ECM_FB_marker_delta_distances_IHD_summary_plot.pdf")
if figure_path.exists():
    display(IFrame(src=str(figure_path), width="100%", height=600))
else:
    print(f"Original figure not found: {figure_path.resolve()}")

## Run the analysis from the annotated AnnData objects

### Computational definitions used by the code

- **Cell types:** `CM` and `FB` are selected from the final annotation column specified for each dataset.
- **Expression layer:** `log_norm`, generated by total-count normalization to 10,000 followed by `log1p`. If raw counts are available in `layers['counts']`, this layer is regenerated from those counts; otherwise an existing `log_norm` layer is used after validation. If neither is available, `.X` must be raw Xenium counts and is validated before normalization. Validation is performed in chunks for dense matrices to limit memory use.
- **NPPA-positive CM:** CM with `NPPA > 0` in `log_norm`.
- **NPPA-high anchor:** the top 10% of NPPA-positive CM by NPPA expression, using `ceil(n × 0.10)` cells.
- **NPPA-negative anchor:** CM with `NPPA == 0` in `log_norm`.
- **Marker-high / marker-low FB:** top/bottom 10% by expression among marker-positive FB only (`expression > 0`).
- **Marker-negative FB:** FB with expression equal to zero for the marker.
- **Random control:** 100 samples of fibroblasts, without replacement within each sample, each matched to the marker-high group size.
- **Distance:** two-dimensional Euclidean distance to the nearest anchor cell, calculated using `scipy.spatial.cKDTree`.
- **Targeted statistical tests:** one-sided Mann–Whitney U tests (`alternative="less"`) for marker-high versus marker-low/marker-negative groups. For random controls, the p-value is the median of 100 one-sided Mann–Whitney U p-values from matched-size random fibroblast samples. Targeted p-values are adjusted with Benjamini–Hochberg within each dataset and anchor.
- **Broad ECM-marker screen:** one-sided Mann–Whitney U test for marker-high distances being smaller than marker-low distances; Benjamini–Hochberg correction is applied across the genes tested within each dataset.
- **Caveat:** cell-level tests do not model donor/section clustering. Use sample-aware inference if sample IDs and enough independent samples are available.

In [ ]:
from pathlib import Path
import warnings
from IPython.display import display

import numpy as np
import pandas as pd
import scanpy as sc
from anndata import AnnData
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.spatial import cKDTree
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

OUTPUT_DIR = Path("xenium_spatial_proximity_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# These annotation versions and columns match the reported analysis counts.
DATASET_CONFIG = {
    "DCM": {"path": Path("DCM_final.h5ad"), "celltype_col": "celltype_corrected"},
    "IHD": {"path": Path("adata_celltype_final_corrected_v4.h5ad"), "celltype_col": "celltype_final_corrected_v4"},
}

EXPRESSION_LAYER = "log_norm"
TARGETED_MARKERS = ["VCAN", "DCN", "PDGFRA"]
FRACTION = 0.10
# Dataset-specific random-control settings.
RANDOM_SEED_BY_DATASET = {"DCM": 42, "IHD": 42}
N_RANDOM = 100
STATISTIC = "median"
MATRISOME_FILE = Path("Hs_Matrisome_Masterlist_Naba et al_2012.xlsx")
CURATED_FB_MARKERS = ["PDGFRA", "THY1"]

# Reference counts for the exact analysis versions used for the supplied summaries.
EXPECTED_COUNTS = {
    "DCM": {"CM": 66162, "FB": 46406, "NPPA_positive_CM": 39347,
            "NPPA_negative_CM": 26815, "NPPA_high_CM": 3935},
    "IHD": {"CM": 43961, "FB": 27223, "NPPA_positive_CM": 29984,
            "NPPA_negative_CM": 13977, "NPPA_high_CM": 2999},
}


def _validate_matrix(matrix, label, integer_like=False):
    """Validate a potentially large dense or sparse matrix in bounded chunks."""
    if sparse.issparse(matrix):
        chunks = [matrix.data]
    else:
        arr = np.asarray(matrix)
        if arr.size == 0:
            raise ValueError(f"{label} is empty.")
        row_step = max(1, min(2048, arr.shape[0] if arr.ndim else 1))
        if arr.ndim == 0:
            chunks = [arr.reshape(1)]
        else:
            chunks = (arr[i:i + row_step].ravel() for i in range(0, arr.shape[0], row_step))
    n_values = 0
    for values in chunks:
        values = np.asarray(values)
        n_values += values.size
        if not np.isfinite(values).all():
            raise ValueError(f"{label} contains non-finite values.")
        if np.any(values < 0):
            raise ValueError(f"{label} contains negative values.")
        if integer_like and not np.allclose(values, np.rint(values), rtol=0, atol=1e-6):
            raise ValueError(
                f"{label} is not integer-like. Do not normalize an already normalized/log-transformed matrix as raw counts."
            )
    if n_values == 0:
        raise ValueError(f"{label} contains no values.")


In [ ]:
def ensure_log_norm_layer(adata, layer_name=EXPRESSION_LAYER):
    """Create a reproducible log-normalized layer without modifying adata.X."""
    # Prefer raw counts when available so normalization is defined by this notebook.
    if "counts" in adata.layers:
        source = adata.layers["counts"]
        _validate_matrix(source, "adata.layers['counts']", integer_like=True)
    elif layer_name in adata.layers:
        existing = adata.layers[layer_name]
        _validate_matrix(existing, f"adata.layers[{layer_name!r}]")
        return adata
    else:
        source = adata.X
        _validate_matrix(source, "adata.X (used as raw counts)", integer_like=True)

    # Build a minimal AnnData object to avoid duplicating every input layer/coordinate array.
    temp = AnnData(
        X=source.copy(),
        obs=pd.DataFrame(index=adata.obs_names.copy()),
        var=pd.DataFrame(index=adata.var_names.copy()),
    )
    sc.pp.normalize_total(temp, target_sum=1e4)
    sc.pp.log1p(temp)
    adata.layers[layer_name] = temp.X.copy()
    return adata


def get_spatial_coordinates(adata):
    """Return the first two spatial coordinates, in their stored units."""
    if "spatial" in adata.obsm:
        coords = np.asarray(adata.obsm["spatial"], dtype=float)
    elif "coord_xy" in adata.obsm:
        coords = np.asarray(adata.obsm["coord_xy"], dtype=float)
    elif {"x_centroid", "y_centroid"}.issubset(adata.obs.columns):
        coords = adata.obs[["x_centroid", "y_centroid"]].to_numpy(dtype=float)
    else:
        raise KeyError("No spatial coordinates found in obsm['spatial'], obsm['coord_xy'] or x/y centroid columns.")

    if coords.ndim != 2 or coords.shape[0] != adata.n_obs or coords.shape[1] < 2:
        raise ValueError(f"Expected an n_cells × 2+ coordinate matrix; got {coords.shape} for n_obs={adata.n_obs}.")
    coords = coords[:, :2]
    if not np.isfinite(coords).all():
        raise ValueError("Spatial coordinates contain non-finite values; resolve before analysis.")
    return coords


def get_gene_values(adata, gene, layer=EXPRESSION_LAYER):
    """Return one gene's expression vector, allowing case-insensitive gene lookup."""
    lookup = {str(name).upper(): str(name) for name in adata.var_names}
    actual_gene = lookup.get(str(gene).upper())
    if actual_gene is None:
        raise KeyError(f"{gene} not found in adata.var_names.")
    if layer not in adata.layers:
        raise KeyError(f"Layer {layer!r} not found in adata.layers.")
    values = adata[:, actual_gene].layers[layer]
    arr = values.toarray().ravel() if sparse.issparse(values) else np.asarray(values).ravel()
    arr = arr.astype(float, copy=False)
    if not np.isfinite(arr).all():
        raise ValueError(f"Expression values for {actual_gene} contain non-finite values.")
    return arr


def top_fraction_mask(expression, base_mask, fraction=FRACTION):
    """Select ceil(fraction × n) highest-expression cells in base_mask."""
    indices = np.where(base_mask)[0]
    values = expression[indices]
    finite = np.isfinite(values)
    indices, values = indices[finite], values[finite]
    if len(indices) == 0:
        raise ValueError("No finite cells found in the requested anchor population.")
    n_select = max(1, int(np.ceil(len(indices) * fraction)))
    selected = indices[np.argsort(values)[::-1][:n_select]]
    mask = np.zeros(len(base_mask), dtype=bool)
    mask[selected] = True
    return mask


def split_fibroblasts_by_marker(adata, gene, fb_mask, fraction=FRACTION,
                                expression_layer=EXPRESSION_LAYER):
    """Define high/low as top/bottom fractions among detectable FB; zero-expression FB form a third group."""
    expression = get_gene_values(adata, gene, layer=expression_layer)
    fb_indices = np.where(fb_mask)[0]
    fb_values = expression[fb_indices]
    finite = np.isfinite(fb_values)
    fb_indices, fb_values = fb_indices[finite], fb_values[finite]
    positive = fb_values > 0
    positive_indices, positive_values = fb_indices[positive], fb_values[positive]
    zero_indices = fb_indices[fb_values == 0]
    if len(positive_indices) < 2:
        raise ValueError(f"Fewer than two marker-positive fibroblasts are available for {gene}.")
    n_group = max(1, int(np.ceil(len(positive_indices) * fraction)))
    if 2 * n_group > len(positive_indices):
        raise ValueError(f"Too few marker-positive fibroblasts to form distinct high/low groups for {gene}.")
    order = np.argsort(positive_values)
    low_indices = positive_indices[order[:n_group]]
    high_indices = positive_indices[order[-n_group:]]
    high = np.zeros(adata.n_obs, dtype=bool)
    low = np.zeros(adata.n_obs, dtype=bool)
    zero = np.zeros(adata.n_obs, dtype=bool)
    positive_mask = np.zeros(adata.n_obs, dtype=bool)
    high[high_indices] = True
    low[low_indices] = True
    zero[zero_indices] = True
    positive_mask[positive_indices] = True
    return high, low, zero, positive_mask


def nearest_anchor_distance(coords, anchor_mask):
    """2D Euclidean distance from every cell to its nearest anchor cell."""
    anchor_coords = coords[anchor_mask]
    if len(anchor_coords) == 0:
        raise ValueError("Anchor population is empty.")
    return cKDTree(anchor_coords).query(coords, k=1)[0]


def random_fibroblast_mask(fb_mask, n_select, rng):
    fb_indices = np.where(fb_mask)[0]
    if n_select <= 0 or n_select > len(fb_indices):
        raise ValueError("Random control group size must be between 1 and the total number of fibroblasts.")
    selected = rng.choice(fb_indices, size=n_select, replace=False)
    mask = np.zeros_like(fb_mask, dtype=bool)
    mask[selected] = True
    return mask


def summarize_pair(gene, distances, high_mask, reference_mask, reference_label,
                   statistic=STATISTIC):
    """Compare marker-high distances against low/negative groups using a one-sided MW test."""
    reducer = np.median if statistic == "median" else np.mean
    d_high = distances[high_mask]
    d_ref = distances[reference_mask]
    d_high = d_high[np.isfinite(d_high)]
    d_ref = d_ref[np.isfinite(d_ref)]
    record = {
        "gene": gene, "comparison": f"marker high vs {reference_label}",
        "reference_group": reference_label, "n_high": int(len(d_high)),
        "n_reference": int(len(d_ref)), "distance_high": np.nan,
        "distance_reference": np.nan, "delta_high_minus_reference": np.nan,
        "delta_reference_minus_high": np.nan, "p_value": np.nan,
        "test_method": "Mann–Whitney U, one-sided (alternative=less)",
    }
    if len(d_high) == 0 or len(d_ref) == 0:
        return record
    dist_high = float(reducer(d_high))
    dist_ref = float(reducer(d_ref))
    p_value = float(mannwhitneyu(d_high, d_ref, alternative="less").pvalue)
    record.update({
        "distance_high": dist_high, "distance_reference": dist_ref,
        "delta_high_minus_reference": dist_high - dist_ref,
        "delta_reference_minus_high": dist_ref - dist_high,
        "p_value": p_value,
    })
    return record


In [ ]:
def run_targeted_marker_analysis(adata, dataset_name, celltype_col):
    """Run VCAN/DCN/PDGFRA comparisons for NPPA-high and NPPA-negative CM anchors."""
    if celltype_col not in adata.obs.columns:
        raise KeyError(f"Cell-type column {celltype_col!r} not found in adata.obs.")
    if "NPPA" not in {str(g).upper() for g in adata.var_names}:
        raise KeyError("NPPA not found in adata.var_names.")

    coords = get_spatial_coordinates(adata)
    celltype = adata.obs[celltype_col].astype(str).to_numpy()
    cm_mask = celltype == "CM"
    fb_mask = celltype == "FB"
    if not cm_mask.any() or not fb_mask.any():
        raise ValueError(f"{dataset_name}: final annotation must contain non-zero CM and FB populations.")
    nppa = get_gene_values(adata, "NPPA")
    nppa_positive_cm = cm_mask & (nppa > 0)
    nppa_negative_cm = cm_mask & (nppa == 0)
    nppa_high_cm = top_fraction_mask(nppa, nppa_positive_cm, FRACTION)

    count_row = {
        "dataset": dataset_name, "n_cells_total": int(adata.n_obs),
        "n_CM": int(cm_mask.sum()), "n_FB": int(fb_mask.sum()),
        "n_NPPA_positive_CM": int(nppa_positive_cm.sum()),
        "n_NPPA_negative_CM": int(nppa_negative_cm.sum()),
        "n_NPPA_high_CM_anchor": int(nppa_high_cm.sum()),
        "celltype_column": celltype_col, "expression_layer": EXPRESSION_LAYER,
        "fraction": FRACTION,
    }

    anchors = [
        (nppa_high_cm, "NPPA top 10% of NPPA-positive CM"),
        (nppa_negative_cm, "NPPA-negative CM"),
    ]
    rows = []
    gene_lookup = {str(g).upper() for g in adata.var_names}
    missing_markers = [g for g in TARGETED_MARKERS if g.upper() not in gene_lookup]
    if missing_markers:
        raise KeyError(f"{dataset_name}: required target markers missing: {missing_markers}")

    seed = RANDOM_SEED_BY_DATASET[dataset_name]
    rng = np.random.default_rng(seed)
    print(f"{dataset_name}: random-control seed = {seed!r}")
    for anchor_mask, anchor_name in anchors:
        distances = nearest_anchor_distance(coords, anchor_mask)
        for gene in TARGETED_MARKERS:
            high, low, zero, positive = split_fibroblasts_by_marker(adata, gene, fb_mask)
            for ref_mask, ref_label in [(low, "marker-low FB"), (zero, "marker-negative FB")]:
                rec = summarize_pair(gene, distances, high, ref_mask, ref_label)
                rec.update({
                    "dataset": dataset_name, "anchor": anchor_name,
                    "n_marker_positive_FB": int(positive.sum()),
                    "n_marker_negative_FB": int(zero.sum()),
                    "fraction": FRACTION, "expression_layer": EXPRESSION_LAYER,
                })
                rows.append(rec)

            # Matched-size random fibroblast baseline (100 draws, no replacement within draw).
            random_medians = []
            random_p_values = []
            d_high = distances[high]
            d_high = d_high[np.isfinite(d_high)]
            for _ in range(N_RANDOM):
                random_mask = random_fibroblast_mask(fb_mask, int(high.sum()), rng)
                d_random = distances[random_mask]
                d_random = d_random[np.isfinite(d_random)]
                random_medians.append(float(np.median(d_random)) if len(d_random) else np.nan)
                if len(d_high) and len(d_random):
                    random_p_values.append(float(mannwhitneyu(d_high, d_random, alternative="less").pvalue))
                else:
                    random_p_values.append(np.nan)

            valid_random = np.asarray(random_medians, dtype=float)
            valid_random = valid_random[np.isfinite(valid_random)]
            valid_random_p = np.asarray(random_p_values, dtype=float)
            valid_random_p = valid_random_p[np.isfinite(valid_random_p)]
            med_high = float(np.median(d_high)) if len(d_high) else np.nan
            if len(valid_random) == 0 or not np.isfinite(med_high):
                p_random = np.nan
                mean_random_median = np.nan
                sd_random = p05_random = p95_random = np.nan
            else:
                mean_random_median = float(np.mean(valid_random))
                sd_random = float(np.std(valid_random, ddof=1)) if len(valid_random) > 1 else 0.0
                p05_random, p95_random = np.percentile(valid_random, [5, 95]).astype(float)
                # Report the median of the 100 one-sided Mann–Whitney U p-values, as in the original analysis.
                p_random = float(np.median(valid_random_p)) if len(valid_random_p) else np.nan

            rows.append({
                "gene": gene, "comparison": "marker high vs random FB",
                "reference_group": "size-matched random FB", "n_high": int(len(d_high)),
                "n_reference": int(high.sum()), "distance_high": med_high,
                "distance_reference": mean_random_median,
                "delta_high_minus_reference": med_high - mean_random_median,
                "delta_reference_minus_high": mean_random_median - med_high,
                "p_value": p_random,
                "test_method": "Median of 100 one-sided Mann–Whitney U tests (alternative=less); matched-size random FB draws",
                "random_distance_sd": sd_random, "random_distance_p05": p05_random,
                "random_distance_p95": p95_random,
                "dataset": dataset_name, "anchor": anchor_name,
                "n_marker_positive_FB": int(positive.sum()),
                "n_marker_negative_FB": int(zero.sum()),
                "fraction": FRACTION, "expression_layer": EXPRESSION_LAYER,
            })

    results = pd.DataFrame(rows)
    results["fdr_bh_within_anchor"] = np.nan
    for anchor_name in results["anchor"].unique():
        valid = (results["anchor"] == anchor_name) & results["p_value"].notna()
        if valid.any():
            results.loc[valid, "fdr_bh_within_anchor"] = multipletests(
                results.loc[valid, "p_value"], method="fdr_bh"
            )[1]
    return count_row, results


In [ ]:
# Load the exact annotated objects used for the supplied summaries. Stop if either input is missing.
all_counts = []
all_targeted_results = []
loaded_objects = {}

missing_inputs = [str(cfg["path"].resolve()) for cfg in DATASET_CONFIG.values() if not cfg["path"].exists()]
if missing_inputs:
    raise FileNotFoundError(
        "Required annotated input object(s) not found. Place both version-matched files beside this notebook:\n- "
        + "\n- ".join(missing_inputs)
    )

for dataset_name, cfg in DATASET_CONFIG.items():
    path = Path(cfg["path"])
    print(f"\nLoading {dataset_name}: {path.resolve()}")
    adata = sc.read_h5ad(path)
    adata = ensure_log_norm_layer(adata)
    counts, results = run_targeted_marker_analysis(
        adata=adata, dataset_name=dataset_name, celltype_col=cfg["celltype_col"]
    )

    expected = EXPECTED_COUNTS[dataset_name]
    observed_keys = {
        "CM": "n_CM", "FB": "n_FB", "NPPA_positive_CM": "n_NPPA_positive_CM",
        "NPPA_negative_CM": "n_NPPA_negative_CM", "NPPA_high_CM": "n_NPPA_high_CM_anchor",
    }
    mismatches = {
        key: {"expected": expected[key], "observed": counts[source_key]}
        for key, source_key in observed_keys.items()
        if int(counts[source_key]) != int(expected[key])
    }
    if mismatches:
        raise ValueError(
            f"{dataset_name} input does not match the analysis version associated with the supplied results: {mismatches}. "
            "Check the annotation object, cell-type column and NPPA expression layer before proceeding."
        )

    all_counts.append(counts)
    all_targeted_results.append(results)
    loaded_objects[dataset_name] = adata

cell_counts_df = pd.DataFrame(all_counts)
cell_counts_df.to_csv(OUTPUT_DIR / "cell_selection_counts.csv", index=False)
display(cell_counts_df)

targeted_results_df = pd.concat(all_targeted_results, ignore_index=True)
targeted_results_df.to_csv(OUTPUT_DIR / "targeted_marker_distance_results.csv", index=False)
display(targeted_results_df.sort_values(["dataset", "anchor", "gene", "comparison"]))


## 2. Optional broad ECM / fibroblast-marker screen

This section reproduces the logic behind the attached `ECM_FB_marker_delta_distances_DCM_summary_plot.pdf` and `ECM_FB_marker_delta_distances_IHD_summary_plot.pdf` files. It selects the Naba matrisome core categories *collagens*, *proteoglycans* and *ECM glycoproteins*, adds `PDGFRA` and `THY1`, and compares marker-high with marker-low fibroblasts for the NPPA-high CM anchor. The workbook `Hs_Matrisome_Masterlist_Naba et al_2012.xlsx` is required for this section; if it is absent, the targeted analysis above still runs.

In [ ]:
def find_column(df, candidates, label):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    raise KeyError(f"Could not find {label} column. Available columns: {list(df.columns)}")


def load_matrisome_core_genes(workbook_path):
    mat = pd.read_excel(workbook_path)
    gene_col = find_column(mat, ["Gene Symbol", "Gene symbol", "gene_symbol", "GeneSymbol", "Symbol", "symbol", "Gene", "gene"], "gene symbol")
    category_col = find_column(mat, ["Matrisome Category", "Matrisome category", "Category", "category"], "matrisome category")
    core_terms = {"collagens", "proteoglycans", "ecm glycoproteins"}
    category = mat[category_col].astype(str).str.strip().str.lower()
    genes = (mat.loc[category.isin(core_terms), gene_col]
             .dropna().astype(str).str.strip().str.upper().unique().tolist())
    return genes, mat, gene_col, category_col


def run_ecm_core_screen(adata, dataset_name, celltype_col, workbook_path=MATRISOME_FILE):
    if not Path(workbook_path).exists():
        raise FileNotFoundError(f"Matrisome workbook not found: {workbook_path}")
    if celltype_col not in adata.obs.columns:
        raise KeyError(f"Cell-type column '{celltype_col}' not found in adata.obs.")

    core_genes, mat, gene_col, category_col = load_matrisome_core_genes(workbook_path)
    var_upper = pd.Index(adata.var_names.astype(str).str.upper())
    present_curated = [g for g in CURATED_FB_MARKERS if g.upper() in var_upper]
    combined = sorted(set(core_genes) | {g.upper() for g in present_curated})
    gene_name_map = {str(g).upper(): str(g) for g in adata.var_names}
    genes = [gene_name_map[g] for g in combined if g in gene_name_map]

    coords = get_spatial_coordinates(adata)
    celltype = adata.obs[celltype_col].astype(str).to_numpy()
    cm_mask = celltype == "CM"
    fb_mask = celltype == "FB"
    nppa = get_gene_values(adata, "NPPA")
    nppa_positive_cm = cm_mask & (nppa > 0)
    nppa_high_cm = top_fraction_mask(nppa, nppa_positive_cm, FRACTION)
    distances = nearest_anchor_distance(coords, nppa_high_cm)

    rows = []
    skipped = []
    for gene in genes:
        try:
            high, low, zero, positive = split_fibroblasts_by_marker(adata, gene, fb_mask)
            d_high, d_low = distances[high], distances[low]
            d_high = d_high[np.isfinite(d_high)]
            d_low = d_low[np.isfinite(d_low)]
            if len(d_high) == 0 or len(d_low) == 0:
                raise ValueError("Empty marker-high or marker-low group.")
            p_value = float(mannwhitneyu(d_high, d_low, alternative="less").pvalue)
            rows.append({
                "dataset": dataset_name,
                "anchor": "NPPA top 10% of NPPA-positive CM",
                "gene": gene,
                "gene_source": "curated_FB_marker" if gene.upper() in {g.upper() for g in present_curated} else "Naba_ECM_core",
                "n_marker_positive_FB": int(positive.sum()),
                "n_marker_negative_FB": int(zero.sum()),
                "n_high": int(len(d_high)),
                "n_low": int(len(d_low)),
                "distance_marker_high_median": float(np.median(d_high)),
                "distance_marker_low_median": float(np.median(d_low)),
                "delta_low_minus_high": float(np.median(d_low) - np.median(d_high)),
                "delta_high_minus_low": float(np.median(d_high) - np.median(d_low)),
                "p_mann_whitney_high_closer_than_low": p_value,
                "expression_layer": EXPRESSION_LAYER,
                "fraction": FRACTION,
            })
        except Exception as exc:
            skipped.append({"dataset": dataset_name, "gene": gene, "reason": str(exc)})

    summary = pd.DataFrame(rows)
    if summary.empty:
        raise ValueError(f"No genes could be tested for {dataset_name}.")
    summary["fdr_bh"] = multipletests(summary["p_mann_whitney_high_closer_than_low"], method="fdr_bh")[1]
    summary = summary.sort_values("delta_low_minus_high", ascending=False).reset_index(drop=True)

    prefix = f"ECM_FB_marker_delta_distances_{dataset_name}"
    summary.to_csv(OUTPUT_DIR / f"{prefix}.csv", index=False)
    if skipped:
        pd.DataFrame(skipped).to_csv(OUTPUT_DIR / f"{prefix}_skipped_genes.csv", index=False)

    fig_width = max(12, len(summary) * 0.22)
    fig, ax = plt.subplots(figsize=(fig_width, 5))
    x = np.arange(len(summary))
    ax.scatter(x, summary["delta_low_minus_high"], s=45, edgecolor="black", linewidth=0.5)
    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(summary["gene"], rotation=90, fontsize=8)
    ax.set_ylabel("Median distance(marker low) − distance(marker high)")
    ax.set_xlabel("ECM core genes + curated FB markers")
    ax.set_title(
        f"{dataset_name}: ECM/FB marker delta distance to NPPA-high CM anchor\n"
        "positive delta = marker-high FB are closer to NPPA-high CM"
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{prefix}_summary_plot.pdf", bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{prefix}_summary_plot.png", dpi=300, bbox_inches="tight")
    plt.show()

    print(f"{dataset_name}: {len(core_genes)} Naba core genes in source list; {len(genes)} combined genes tested/present before any skipped genes.")
    print(f"Saved: {OUTPUT_DIR / f'{prefix}.csv'}")
    return summary

In [ ]:
# Run the broad ECM screen for any loaded dataset when the workbook is available.
if MATRISOME_FILE.exists():
    all_ecm_summaries = []
    for dataset_name, adata in loaded_objects.items():
        celltype_col = DATASET_CONFIG[dataset_name]["celltype_col"]
        ecm_summary = run_ecm_core_screen(
            adata=adata,
            dataset_name=dataset_name,
            celltype_col=celltype_col,
            workbook_path=MATRISOME_FILE,
        )
        all_ecm_summaries.append(ecm_summary)
    if all_ecm_summaries:
        broad_ecm_results_df = pd.concat(all_ecm_summaries, ignore_index=True)
        broad_ecm_results_df.to_csv(OUTPUT_DIR / "ECM_FB_marker_delta_distances_all_datasets.csv", index=False)
else:
    print(f"Skipping broad ECM screen: workbook not found at {MATRISOME_FILE.resolve()}")

## 3. Outputs and reproducibility notes

The notebook writes results to `xenium_spatial_proximity_outputs/`:

- `cell_selection_counts.csv`: CM/FB, NPPA-positive/negative CM and NPPA-high anchor counts.
- `targeted_marker_distance_results.csv`: median distances and statistical comparisons recalculated from the input AnnData objects.
- `reference_figures/targeted_marker_distance_values_as_reported.csv` and `targeted_marker_distance_summary.pdf/.png`: the medians supplied in the Results text, with the summary figure.
- `targeted_marker_distance_results.csv`: targeted marker distances, deltas, test type, p-values and Benjamini–Hochberg-adjusted p-values for the NPPA-high and NPPA-negative anchors.
- `ECM_FB_marker_delta_distances_<DATASET>.csv`: broad ECM-marker distance summary, including the one-sided test and FDR-adjusted values.
- `ECM_FB_marker_delta_distances_<DATASET>_summary_plot.pdf` and `.png`: recreated broad ECM summary figures using the supplied figure sign convention.
- `ECM_FB_marker_delta_distances_<DATASET>_skipped_genes.csv` (if needed): genes omitted from the broad screen and the corresponding reason.

Both annotated input objects and their annotation columns are required; the analysis stops when an object is missing or the CM, FB or NPPA anchor counts do not match the reported analysis version. If raw counts are available in `layers['counts']`, `log_norm` is regenerated from them. Otherwise, an existing `log_norm` layer is used after basic value validation, or `.X` is assumed to contain raw counts and is validated before normalization. The broad ECM screen additionally requires `Hs_Matrisome_Masterlist_Naba et al_2012.xlsx`.

The reported targeted medians and deltas are included for reference. The analysis cells recalculate the statistics from the annotated objects. Tests are performed at the cell level and do not model within-donor or within-section dependence.

**Important source-alignment note.** The original HTML exports contain multiple analysis states. The notebook retains the analysis code selected for this workflow; numerical agreement with all reported medians has not been established.